# AVeriTeC Data Walkthrough

This notebook explores the AVeriTeC development data used by the retrieval
experiments. It uses the same claim loader and candidate-generation
implementation as the main experiment so that the exploratory analysis
reflects the actual experimental inputs.

The walkthrough examines five development claims and asks:

1. What information is supplied for each claim?
2. How is each claim-specific knowledge store represented?
3. Are the annotated evidence sources present in the corresponding knowledge store?
4. What do the sentence-level retrieval candidates look like?
5. What would larger multi-sentence retrieval units look like?

The final experiment uses sentence-level candidates. The larger passage
examples at the end of this notebook are exploratory only.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from fact_verification.data import CandidateStore, load_claims
from fact_verification.evaluation import normalise_url


def find_git_repository(start: Path) -> Path:
    """Find the nearest parent directory containing .git."""

    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError("Could not find the project Git repository.")


PROJECT_ROOT = find_git_repository(Path.cwd())

# Load the local configuration created by notebook 0.
load_dotenv(PROJECT_ROOT / ".env")

configured_root = os.environ.get("AVERITEC_ROOT")

if configured_root:
    AVERITEC_ROOT = Path(configured_root).expanduser().resolve()
else:
    # Fallback to the canonical local location established by notebook 0.
    local_data_root = PROJECT_ROOT / ".local_data" / "averitec"

    if not local_data_root.exists():
        raise RuntimeError("AVERITEC_ROOT is not configured and the local AVeriTeC data directory does not exist.")

    dataset_roots = sorted([path for path in local_data_root.iterdir() if path.is_dir()])

    if len(dataset_roots) != 1:
        raise RuntimeError("AVERITEC_ROOT is not configured and a unique local dataset root could not be resolved.")

    AVERITEC_ROOT = dataset_roots[0]

DEV_PATH = AVERITEC_ROOT / "data" / "dev.json"
TRAIN_PATH = AVERITEC_ROOT / "data" / "train.json"
KNOWLEDGE_STORE_ROOT = AVERITEC_ROOT / "knowledge_store" / "dev" / "output_dev"

assert DEV_PATH.exists(), f"Missing development annotations: {DEV_PATH}"
assert TRAIN_PATH.exists(), f"Missing training annotations: {TRAIN_PATH}"
assert KNOWLEDGE_STORE_ROOT.exists(), f"Missing extracted development knowledge store: {KNOWLEDGE_STORE_ROOT}"

os.environ["AVERITEC_ROOT"] = str(AVERITEC_ROOT)

pd.set_option("display.max_colwidth", 250)

print("Project root:", PROJECT_ROOT)
print("AVeriTeC root:", AVERITEC_ROOT)
print("AVeriTeC revision:", os.environ.get("AVERITEC_REVISION"))
print("Development data:", DEV_PATH)
print("Training data:", TRAIN_PATH)
print("Knowledge store:", KNOWLEDGE_STORE_ROOT)

Possible development annotation files:
[0] C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a\data\dev.json
[1] C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\dev.json
Project root:        C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
AVeriTeC root:       C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a
Development data:    C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\2ca9dee23a2a\data\dev.json
Training data:       C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks\.local_data\averitec\

## 1.1. Inspect Development Claims

Five fixed development claims are inspected so that the walkthrough covers more than one example while remaining reproducible.

In [ ]:
claims = load_claims(root=AVERITEC_ROOT, split="dev")
assert len(claims) == 500

print(f"Loaded {len(claims):,} development claims.")
EXAMPLE_CLAIM_IDS = [0, 1, 2, 3, 4]

example_claims = (claims[claims["claim_id"].isin(EXAMPLE_CLAIM_IDS)].copy().sort_values("claim_id").reset_index(drop=True))

display(example_claims[["claim_id", "claim", "label", "justification"]])

Loaded 500 development claims.
CLAIM
-----
In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.

LABEL
-----
Refuted

JUSTIFICATION
-------------
The answer and sources show that the claim was published in a fake news site so the claim is refuted.


## 1.2. Inspect Annotated Evidence

AVeriTeC represents evidence through questions and annotated answers. Each
answer may identify a source URL. These source URLs later form the basis of
the source-level retrieval evaluation.

In [ ]:
gold_rows = []

for _, claim_row in example_claims.iterrows():
    claim_id = int(claim_row["claim_id"])
    questions = (claim_row.get("questions") or [])

    for question_number, question in enumerate(questions, start=1):
        answers = (question.get("answers") or [])

        if not answers:
            gold_rows.append({"claim_id": claim_id, "question_number": question_number, "answer_number": None, "question": question.get("question"), 
                              "answer": None, "answer_type": None, "source_url": None, "cached_source_url": None})
            continue

        for answer_number, answer in enumerate(answers, start=1):
            gold_rows.append({"claim_id": claim_id, "question_number": question_number, "answer_number": answer_number, "question": question.get("question"), 
                              "answer": answer.get("answer"), "answer_type": answer.get("answer_type"), "source_url": answer.get("source_url"), 
                              "cached_source_url": answer.get("cached_source_url")})

gold_evidence = pd.DataFrame(gold_rows)

print("Annotated questions:", gold_evidence[["claim_id", "question_number"]].drop_duplicates().shape[0],)
print("Annotated answers:", gold_evidence["answer"].notna().sum())
display(gold_evidence)

Annotated questions: 2
Annotated answers: 2


,question_number,answer_number,question,answer,answer_type,source_url,cached_source_url
0,1,1,Where was the claim first published,It was first published on Sccopertino,Abstractive,https://web.archive.org/web/20201129141238/https://scoopertino.com/exposed-the-imac-disaster-that-almost-was/,https://web.archive.org/web/20201129141238/https://scoopertino.com/exposed-the-imac-disaster-that-almost-was/
1,2,1,What kind of website is Scoopertino,"Scoopertino is an imaginary news organization devoted to ferreting out the most relevant stories in the world of Apple, whether or not they actually occurred - says their about page",Extractive,https://web.archive.org/web/20201202085933/https://scoopertino.com/about-scoopertino/,https://web.archive.org/web/20201202085933/https://scoopertino.com/about-scoopertino/


## 2. Load Claim-Specific Candidate Evidence

Candidate evidence is generated using the same `CandidateStore` used by the
main retrieval experiment. The production experiment uses individual
sentences as retrieval units.

In [ ]:
candidate_store = CandidateStore(root=AVERITEC_ROOT, split="dev", retrieval_unit="sentence", chunk_size=None, chunk_overlap=None)
assert len(candidate_store) == 500

candidate_collections = {}
for claim_id in EXAMPLE_CLAIM_IDS:
    candidates = candidate_store.load_claim(claim_id)

    candidate_collections[claim_id] = candidates
    print(f"Claim {claim_id}: {len(candidates):,} candidates, {candidates['source_url'].nunique():,} unique source URLs")


print("\nCandidate-store directory:", candidate_store.directory)

## 2.1. Inspect Candidate Representation

Each sentence candidate retains its source URL and sentence position so that
retrieval can operate at sentence level while evaluation can still identify
the Web source from which the sentence originated.

In [ ]:
EXAMPLE_CLAIM_ID = (EXAMPLE_CLAIM_IDS[0])

example_candidates = (candidate_collections[EXAMPLE_CLAIM_ID])


print(f"Claim ID: {EXAMPLE_CLAIM_ID}")

print(f"Candidates generated: {len(example_candidates):,}")


display(example_candidates[["candidate_id", "source_url", "text", "sentence_start", "sentence_end"]].head(20))

## 3. Gold Sources in the Candidate Collection

The annotated source URLs are compared with the source URLs represented in
the generated sentence candidates. URL comparison uses the same normalisation
function as the retrieval evaluation rather than requiring exact raw-string
equality.

In [ ]:
coverage_rows = []

for claim_id in EXAMPLE_CLAIM_IDS:
    candidates = candidate_collections[claim_id]

    candidate_urls = {normalise_url(url) for url in candidates["source_url"].dropna()}
    candidate_urls.discard(None)

    claim_gold = gold_evidence[gold_evidence["claim_id"] == claim_id].copy()

    for _, row in claim_gold.iterrows():
        source_url = row["source_url"]

        if pd.notna(source_url) and str(source_url).strip():
            normalised_gold_url = normalise_url(source_url)
        else:
            normalised_gold_url = None

        coverage_rows.append({
            "claim_id": claim_id,
            "question_number": row["question_number"],
            "answer_number": row["answer_number"],
            "source_url": source_url,
            "normalised_source_url": normalised_gold_url,
            "source_present": normalised_gold_url in candidate_urls if normalised_gold_url else False,
        })

gold_source_coverage = pd.DataFrame(coverage_rows)

display(gold_source_coverage)

# Source-level retrieval evaluates unique valid evidence sources rather than
# individual annotated answers.
unique_gold_source_coverage = (
    gold_source_coverage
    .dropna(subset=["normalised_source_url"])
    .drop_duplicates(subset=["claim_id", "normalised_source_url"])
    .reset_index(drop=True)
)

coverage_summary = (
    unique_gold_source_coverage
    .groupby("claim_id")["source_present"]
    .agg(
        annotated_sources="count",
        sources_present="sum",
    )
)

coverage_summary["coverage"] = coverage_summary["sources_present"] / coverage_summary["annotated_sources"]

display(coverage_summary)

### 3.1 Inspect Sentences from a Gold Source

In [ ]:
available_gold_sources = unique_gold_source_coverage[unique_gold_source_coverage["source_present"]]
if available_gold_sources.empty:
    raise RuntimeError("None of the inspected gold sources were present in the candidate store.")

selected_gold = (available_gold_sources.iloc[0])
selected_claim_id = int(selected_gold["claim_id"])
selected_gold_url = (selected_gold["normalised_source_url"])

selected_candidates = (candidate_collections[selected_claim_id].copy())
selected_candidates["normalised_source_url"] = (selected_candidates["source_url"].map(normalise_url))

gold_source_sentences = (selected_candidates[selected_candidates["normalised_source_url"] == selected_gold_url].copy())

print("Claim ID:", selected_claim_id)
print("Gold source:", selected_gold["source_url"])
print("Candidate sentences:", len(gold_source_sentences))

display(gold_source_sentences[["candidate_id", "sentence_start", "sentence_end", "text"]].head(30))

## 4. Walkthrough Summary

The final retrieval experiments use individual sentences as retrieval candidates while retaining the originating source URL for source-level evaluation.

Each claim is associated with its own AVeriTeC knowledge store, from which sentence candidates are constructed using a common representation across all retrieval methods. This ensures that differences in retrieval effectiveness are attributable to the retrieval architecture rather than changes in the underlying candidate evidence.

The annotated evidence inspection also confirms the relationship between AVeriTeC gold source URLs and the source metadata retained by the candidate representation, which forms the basis of the source-level retrieval metrics used in the experimental evaluation.